# 02 — Одиночный эксперимент: оконный сигнал и бэктестер

По умолчанию полностью выполняется на синтетике без Google Cloud и паролей. Реализованы max/quantile/last окна и no-move. Прогнозные модели пока не реализованы.

Цикл: завершить предыдущие переносы → оценить текущую нагрузку → принять решение только по history < t → добавить текущее наблюдение. Решение в t влияет на t+1+duration. При duration=0 дополнительного времени в пути нет.


In [ ]:
from pathlib import Path
from IPython.display import display
from vmforecast import __version__
from vmforecast.settings import Settings

SETTINGS = Settings.load()
ROOT = SETTINGS.root
LAB_DATA_DIR = SETTINGS.data
LAB_RUNS_DIR = SETTINGS.runs
print('vmforecast-lab', __version__)


In [ ]:
from vmforecast import make_synthetic, WindowSignal, SimulationConfig
from vmforecast.data import load_workload
from vmforecast.experiments import run_saved
from vmforecast.reporting import single_report
import json
USE_GOOGLE = False
WARMUP = 8
if USE_GOOGLE:
    workload = load_workload(LAB_DATA_DIR/'google2019_a_6h_20'/'prepared')
    WARMUP = 12
else:
    workload = make_synthetic(seed=7, ticks=96, services=8, hosts=3,
                              warmup=WARMUP, scenario='returning_burst')
signal = WindowSignal(window=12, mode='max')
config = SimulationConfig(warmup=WARMUP,price=.03,horizon=1,
                          migration_duration_ticks=0,max_concurrent=3)
summary, run_dir = run_saved(workload,signal,config,LAB_RUNS_DIR/'single')
print(json.dumps({k:v for k,v in summary.items() if k!='artifact_hashes'},indent=2))
print('Run directory:',run_dir)


In [ ]:
report = single_report(run_dir)
from IPython.display import display, Image, HTML
from pathlib import Path
display(Image(filename=str(run_dir/'timeseries.png')))
print('Interactive report:',report)


## Интерпретация
`violation_rate` — число перегруженных host-ticks / число host-ticks всего фиксированного парка на test. CPU+RAM превышение на одном хосте считается одним нарушением. Это proxy ресурсов, не измеренный HTTP SLA.

`risk_sum` — сумма квадратов положительных нормированных превышений. CPU/RAM excess integrals сохраняются отдельно. Цена условная, не рубли. Миграции не создают/удаляют сервисы. При нулевой цене выбирается перенос только при строгом уменьшении ожидаемого риска.

При длительности миграции >0 нагрузка остаётся на source до завершения; overhead задаётся долей capacity каждого endpoint. Это явно заданная модель, а не измерение физической VM migration.


In [ ]:
# Inspect the exact actions and validate that the world was not modified.
import pandas as pd
display(pd.read_parquet(run_dir/'migrations.parquet').head(20))
assert summary['workload_hash'] == workload.fingerprint()
print('Input workload unchanged; per-tick and per-host metrics saved.')
